<a href="https://colab.research.google.com/github/eric8765/transformer-from-scratch/blob/main/RoPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install jaxtyping

In [ ]:
import torch as t
import einops
from jaxtyping import Float
from types import SimpleNamespace

In [ ]:
cfg = SimpleNamespace(
    d_model = 512,
    n_head = 8,
    base = 10000,
    seq_len = 64
)

In [ ]:
def RoPE(cfg: SimpleNamespace) -> Float[t.Tensor, "seq_len d_model d_model"]:
  d_head = cfg.d_model // cfg.n_head
  pairings = d_head // 2

  m = t.zeros((cfg.seq_len, d_head, d_head))

  k = t.arange(1, cfg.seq_len+1, 1).unsqueeze(-1) #(seq_len, 1)
  degree = k / t.exp(t.arange(0, d_head, 2, dtype = t.float32) / d_head * t.log(t.tensor(cfg.base))) #(seq_len, pairings)

  cos = t.cos(einops.repeat(degree, "seq pair -> seq (pair 2)"))
  sin = t.sin(degree)

  m_diag = t.diagonal(m, dim1=1, dim2=2)
  m_diag[:] = cos

  m_diag_1u = t.diagonal(m, dim1=1, dim2=2, offset=1)
  m_diag_1u[:,0::2] = -1*sin

  m_diag_1d = t.diagonal(m, dim1=1, dim2=2, offset=-1)
  m_diag_1d[:,0::2] = sin

  return m

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(cfg.d_model, cfg.d_model))
plt.imshow(RoPE(cfg)[0], cmap='viridis')
plt.colorbar()
plt.title("1st token RoPE")
plt.show()